In [1]:
import osmnx as ox
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
import folium
from shapely.geometry import LineString, Point
from shapely.ops import unary_union
import networkx as nx
import os
from branca.colormap import LinearColormap

In [2]:
operation_crs = "EPSG:2056"  # Swiss coordinate system
target_crs = "EPSG:4326"  # WGS84 coordinate system

input_file_path = '../../Data/input'
output_step1_path='../../Data/output/step-1'
output_step2_path='../../Data/output/step-2'
output_step3_path='../../Data/output/step-3'

#import file
print("Import files ...")
pedestrian_network = gpd.read_parquet(f'{output_step1_path}/step1_pedestrian_segments.parquet')

index_scores = gpd.read_parquet(f'{output_step3_path}/step3_index.parquet')


Import files ...


In [3]:
#Check colomns
index_scores.head()


,segment_id,geometry,length,bruit,temperature,conflit_usage,canopee,lac_cours_deau,fontaines,espaces_ouverts,...,zone_apaisee,zone_pietonne,vitesse,walk_index,walk_index_unweighted,indice_marchabilite,Classe_Commodité,Classe_Attractivité,Classe_Infrastructure,Classe_Sécurité
0,000000,"LINESTRING (6.09574 46.18116, 6.09576 46.18114)",2.682,0.0,1.0,0.2500,0.079,0.0,0.0,0.0,...,0.118,0.0,0.118,0.3409,0.3409,0.3409,0.3322,0.1667,0.4882,0.3041
1,000001,"LINESTRING (6.14937 46.17581, 6.14939 46.17584...",6.863,0.0,1.0,0.2500,0.523,0.0,0.0,0.0,...,0.012,0.0,0.012,0.3328,0.3328,0.3328,0.4432,0.1667,0.4600,0.1938
2,000002,"LINESTRING (6.16722 46.20953, 6.16721 46.20956...",27.086,0.0,1.0,0.2500,0.000,1.0,0.0,1.0,...,0.000,0.0,0.000,0.5140,0.5140,0.5140,0.3125,0.5000,0.7881,0.0000
3,000003,"LINESTRING (6.16732 46.20941, 6.16729 46.20945...",15.077,0.0,1.0,0.2500,0.000,1.0,0.0,1.0,...,0.000,0.0,0.000,0.4269,0.4269,0.4269,0.3125,0.5000,0.5568,0.0000
4,000004,"LINESTRING (6.09883 46.20446, 6.09883 46.20445...",4.338,0.0,1.0,0.1668,0.008,0.0,0.0,0.0,...,0.000,0.0,0.000,0.3684,0.3684,0.3684,0.2937,0.2083,0.7117,0.0803


In [4]:
# Check if 'segment_id' exists in both
assert 'segment_id' in pedestrian_network.columns
assert 'segment_id' in index_scores.columns

# Convert both to string with 6-digit zero-padding
pedestrian_network['segment_id'] = pedestrian_network['segment_id'].astype(str).str.zfill(6)
index_scores['segment_id'] = index_scores['segment_id'].astype(int).astype(str).str.zfill(6)



In [5]:
# Merge
print("Add column (containing features count) to merge -->")
pedestrian_network = pedestrian_network.merge(index_scores[['segment_id', 'I-Zscore', 'N-ProxAmenities', 'N-RoadSafety']], on='segment_id', how='left')


Add column (containing features count) to merge -->


KeyError: "['I-Zscore', 'N-ProxAmenities', 'N-RoadSafety'] not in index"

In [ ]:
# Load pedestrian network attributes
# From SITG (shp)
print("Loading accidents involving pedestrians")
file_path = '/Users/Helo/Documents/Action_situee/index-marchabilite-ge/Data/input/accidents/OCT_ACCIDENTS-SHP'
accidents = gpd.read_file(f"{file_path}/OCT_ACCIDENTS.shp")

# Filter accidents that involve pedestrians
accidents["has_pedestrian"] = accidents["PIETONS"] > 0

accidents_pieton = accidents[accidents["has_pedestrian"] == True]

# Define the area (can be the city or the full canton boundary if you have it)
area_name = "Geneva, Switzerland"

# Load amenities GeoDataFrame from OSM
print("Loading amenities...")

tags = {
    "amenity": [
        "school", "university", "hospital", "clinic", "pharmacy",
        "restaurant", "cafe", "bar", "library", "bank", "post_office",
        "kindergarten", "theatre", "cinema", "place_of_worship"
    ]
}

# Download amenities
print(f"Fetching amenities for {area_name}")
amenities = ox.features_from_place(area_name, tags=tags)

In [ ]:
pedestrian_network_wgs = pedestrian_network.to_crs(epsg=4326)
accidents_pieton_wsg = accidents_pieton.to_crs(epsg=4326)
amenities_wsg = amenities.to_crs(epsg=4326)

# Define the colormap
import branca.colormap as cm

colormap = cm.linear.RdYlGn_09.scale(0, 1)
colormap.caption = 'Score Normalisé (0 = Mauvais, 1 = Très Bon)'

colormap_acc = LinearColormap(
    colors=["#E60000", "#ffffff"],  # dark red → white
    vmin=0, vmax=1
)
colormap_acc.caption = 'Score Normalisé (0 = Mauvais, 1 = Très Bon)'


def make_style_function(column_name, colormap):
    def style_function(feature):
        value = feature['properties'].get(column_name, None)
        return {
            'color': colormap(value) if value is not None else 'gray',
            'weight': 5,
            'opacity': 1
        }
    return style_function

import branca.colormap as cm


# Generate custom style functions
style_accident = make_style_function("N-RoadSafety", colormap_acc)
style_amenities = make_style_function("N-ProxAmenities", colormap)
style_index = make_style_function("I-Zscore", colormap)


In [ ]:
# Compute center
center = pedestrian_network_wgs.geometry.unary_union.centroid
m = folium.Map(location=[center.y, center.x], zoom_start=13, tiles="cartodbpositron")

# --- Add pedestrian segments ---
print("Map network")
'''folium.GeoJson(
    pedestrian_network_wgs,
    name='Pedestrian Network',
    style_function=lambda x: {'color': '#3489db', 'weight': 2},
).add_to(m)'''

# ----- Mapping scores ----
# Add GeoJson layer
print("Mapping roadsafety ... ")
folium.GeoJson(
    pedestrian_network_wgs,
    name="Score accidents avec piétons",
    style_function=style_accident,
    tooltip=folium.GeoJsonTooltip(fields=["segment_id", "N-RoadSafety"], aliases=["ID", "Score Accidents"])
).add_to(m)

# Add GeoJson layer
print("Mapping prox amenities ... ")
folium.GeoJson(
    pedestrian_network_wgs,
    name="Score proximité aménités",
    style_function=style_amenities,
    tooltip=folium.GeoJsonTooltip(fields=["segment_id", "N-ProxAmenities"], aliases=["ID", "Score Proximité Amenités"])
).add_to(m)

# Add GeoJson layer
print("Mapping global index ... ")
folium.GeoJson(
    pedestrian_network_wgs,
    name="Indice de marchabilité",
    style_function=style_index,
    tooltip=folium.GeoJsonTooltip(fields=["segment_id", "I-Zscore"], aliases=["ID", "Indice"])
).add_to(m)

# ----- Mapping features -----
'''# Create a FeatureGroup for accidents
accidents_layer = folium.FeatureGroup(name="Accidents piétons", show=True)

print("Mapping accidents...")
for _, row in accidents_pieton_wsg.iterrows():
    folium.CircleMarker(
        location=[row.geometry.y, row.geometry.x],
        radius=1,
        color='#1D3557',
        fill=True,
        fill_opacity=0.7,
        popup='Accident',
    ).add_to(accidents_layer)'''



# Create a FeatureGroup for amenities
'''amenities_layer = folium.FeatureGroup(name="Amenités", show=True)

# Loop through amenities and add them as small green markers
print("Mapping amenities...")
for _, row in amenities_wsg.iterrows():
    amenity_type = row.get("amenity", "Amenity")

    # Use point geometry or fallback to centroid
    geom = row.geometry
    if geom.geom_type == "Point":
        lat, lon = geom.y, geom.x
    else:
        centroid = geom.centroid
        lat, lon = centroid.y, centroid.x

    folium.CircleMarker(
        location=[lat, lon],
        radius=1,
        color='#567572',
        fill=True,
        fill_opacity=0.7,
        popup=folium.Popup(f"Type d'amenité: {amenity_type}", max_width=200),
    ).add_to(amenities_layer)

# Add the FeatureGroup to the map
accidents_layer.add_to(m)
amenities_layer.add_to(m)'''


# ---- Add legend -----
colormap.add_to(m)
colormap_acc.add_to(m)


# Show or save
folium.LayerControl().add_to(m)
#m.save("score_map.html")
m
